# 4.5 Prefix Caching Lab[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.5_prefix_caching/lab.ipynb)[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.5_prefix_caching/lab.ipynb)Benchmark prefix caching: measure TTFT with and without KV reuse for shared system prompts.

In [ ]:
# ============================================================# Setup: clone repo utilities, install dependencies# ============================================================import subprocess, sys, os# Clone repo for shared utilities (persists across cells)if not os.path.exists("/tmp/lis-repo"):    subprocess.run(["git", "clone", "--depth=1",                    "https://github.com/harshuljain13/llm-inference-at-scale.git",                    "/tmp/lis-repo"], check=True)sys.path.insert(0, "/tmp/lis-repo")# Install vLLM (required for prefix caching benchmark)subprocess.run([sys.executable, "-m", "pip", "install", "-q",                "vllm>=0.4.0", "matplotlib", "numpy"], check=True)

## Experiment 1: Manual KV Reuse SimulationWe simulate prefix caching savings by measuring prefill time for the full sequence vs. suffix-only.This works on any GPU (including Colab T4) using HuggingFace transformers.

In [ ]:
# ============================================================# Parameters: adjust these to change the experiment# ============================================================MODEL_NAME = "mistralai/Mistral-7B-v0.1"  # Non-gated, no auth neededSYSTEM_PROMPT_TOKENS = 512   # Length of simulated system promptUSER_QUERY_TOKENS = 64       # Length of simulated user queryNUM_TRIALS = 5               # Repetitions for timing stabilityDEVICE = "cuda"              # Use "cpu" if no GPU available

In [ ]:
import torchimport timeimport numpy as npfrom transformers import AutoModelForCausalLM, AutoTokenizer# Load model and tokenizer (downloads ~14GB on first run)print(f"Loading {MODEL_NAME}...")tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)model = AutoModelForCausalLM.from_pretrained(    MODEL_NAME,    torch_dtype=torch.float16,    device_map="auto",       # Automatic GPU placement    use_cache=True           # Enable KV cache returns)model.eval()print(f"Model loaded on {next(model.parameters()).device}")

In [ ]:
# ============================================================# Generate synthetic input: system prompt + user query# ============================================================# Use real tokens from tokenizer vocabularyvocab_size = tokenizer.vocab_sizetorch.manual_seed(42)# Create fixed system prompt tokens and varying user query tokenssystem_ids = torch.randint(100, vocab_size - 100, (1, SYSTEM_PROMPT_TOKENS)).to(DEVICE)user_ids = torch.randint(100, vocab_size - 100, (1, USER_QUERY_TOKENS)).to(DEVICE)full_ids = torch.cat([system_ids, user_ids], dim=1)print(f"System prompt: {SYSTEM_PROMPT_TOKENS} tokens")print(f"User query: {USER_QUERY_TOKENS} tokens")print(f"Total input: {full_ids.shape[1]} tokens")

In [ ]:
# ============================================================# Benchmark: Full prefill (no caching) vs suffix-only (with cached prefix)# ============================================================def measure_prefill_time(input_ids, past_key_values=None, num_trials=5):    """Measure average prefill time over multiple trials."""    times = []    for _ in range(num_trials):        torch.cuda.synchronize()        start = time.perf_counter()        with torch.no_grad():            outputs = model(                input_ids=input_ids,                past_key_values=past_key_values,                use_cache=True            )        torch.cuda.synchronize()        times.append(time.perf_counter() - start)    return np.array(times), outputs# Warmup run (first run includes CUDA kernel compilation overhead)_ = measure_prefill_time(full_ids, num_trials=1)# Measure FULL prefill: system + user from scratchfull_times, _ = measure_prefill_time(full_ids, num_trials=NUM_TRIALS)# Compute system prompt KV cache ONCE (simulates cached prefix)with torch.no_grad():    prefix_out = model(input_ids=system_ids, use_cache=True)cached_kv = prefix_out.past_key_values  # This is the "cached" prefix# Measure SUFFIX-ONLY prefill: only user query, reusing cached KVsuffix_times, _ = measure_prefill_time(user_ids, past_key_values=cached_kv, num_trials=NUM_TRIALS)print(f"Full prefill ({full_ids.shape[1]} tokens):    {full_times.mean()*1000:.1f} ms (std {full_times.std()*1000:.1f} ms)")print(f"Suffix only ({USER_QUERY_TOKENS} tokens):     {suffix_times.mean()*1000:.1f} ms (std {suffix_times.std()*1000:.1f} ms)")print(f"TTFT speedup:                   {full_times.mean()/suffix_times.mean():.2f}x")print(f"Time saved per request:          {(full_times.mean()-suffix_times.mean())*1000:.1f} ms")

## Experiment 2: Scaling Prefix LengthMeasure how TTFT improvement scales with the length of the cached prefix.

In [ ]:
# ============================================================# Sweep prefix lengths: measure speedup at each# ============================================================import matplotlib.pyplot as pltprefix_lengths = [128, 256, 512, 1024, 2048]suffix_len = 64  # Fixed user query lengthspeedups = []full_ttfts = []cached_ttfts = []for plen in prefix_lengths:    # Generate input    p_ids = torch.randint(100, vocab_size - 100, (1, plen)).to(DEVICE)    s_ids = torch.randint(100, vocab_size - 100, (1, suffix_len)).to(DEVICE)    combined = torch.cat([p_ids, s_ids], dim=1)    # Full prefill    torch.cuda.synchronize()    t0 = time.perf_counter()    with torch.no_grad():        model(input_ids=combined, use_cache=True)    torch.cuda.synchronize()    full_t = time.perf_counter() - t0    # Cached prefix + suffix only    with torch.no_grad():        pout = model(input_ids=p_ids, use_cache=True)    torch.cuda.synchronize()    t0 = time.perf_counter()    with torch.no_grad():        model(input_ids=s_ids, past_key_values=pout.past_key_values, use_cache=True)    torch.cuda.synchronize()    suffix_t = time.perf_counter() - t0    speedups.append(full_t / suffix_t)    full_ttfts.append(full_t * 1000)    cached_ttfts.append(suffix_t * 1000)    print(f"Prefix {plen:>5} tokens: full={full_t*1000:.1f}ms, cached={suffix_t*1000:.1f}ms, speedup={full_t/suffix_t:.2f}x")

In [ ]:
# ============================================================# Plot: TTFT with and without prefix caching# ============================================================fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))# Left: absolute TTFT comparisonx = range(len(prefix_lengths))width = 0.35ax1.bar([i - width/2 for i in x], full_ttfts, width, label="No Cache (full prefill)", color="#ffe4e6", edgecolor="#000")ax1.bar([i + width/2 for i in x], cached_ttfts, width, label="With Prefix Cache", color="#dcfce7", edgecolor="#000")ax1.set_xlabel("Cached Prefix Length (tokens)")ax1.set_ylabel("TTFT (ms)")ax1.set_title("Time-to-First-Token: Full vs Cached Prefill")ax1.set_xticks(x)ax1.set_xticklabels(prefix_lengths)ax1.legend()ax1.grid(axis="y", alpha=0.3)# Right: speedup factorax2.plot(prefix_lengths, speedups, "o-", color="#2563eb", linewidth=2, markersize=8)ax2.axhline(y=1, color="#991b1b", linestyle="--", alpha=0.5, label="No improvement")ax2.set_xlabel("Cached Prefix Length (tokens)")ax2.set_ylabel("TTFT Speedup (x)")ax2.set_title("Speedup from Prefix Caching")ax2.legend()ax2.grid(alpha=0.3)plt.tight_layout()plt.savefig("prefix_caching_benchmark.png", dpi=150, bbox_inches="tight")plt.show()print("Saved: prefix_caching_benchmark.png")

## Experiment 3: vLLM Prefix Caching (requires A10G+ GPU)If running on a GPU with 24GB+ VRAM, this cell benchmarks vLLM's automatic prefix caching end-to-end.

In [ ]:
# ============================================================# vLLM prefix caching benchmark (skip if insufficient VRAM)# ============================================================try:    from vllm import LLM, SamplingParams    # Check available VRAM    free_mem = torch.cuda.mem_get_info()[0] / 1e9    if free_mem < 16:        print(f"Only {free_mem:.1f} GB free VRAM. Need 16GB+ for vLLM. Skipping.")        raise MemoryError("Insufficient VRAM")    # Delete HF model to free memory    del model    torch.cuda.empty_cache()    # System prompt shared across all requests    SYSTEM_PROMPT = "You are a helpful AI assistant. " * 200  # ~800 tokens    # Launch vLLM WITH prefix caching    llm_cached = LLM(        model="mistralai/Mistral-7B-v0.1",        enable_prefix_caching=True,        gpu_memory_utilization=0.85,        max_model_len=4096    )    params = SamplingParams(max_tokens=1, temperature=0)  # 1 token to measure TTFT    # Generate requests with shared prefix    prompts = [SYSTEM_PROMPT + f"Question {i}: What is {i}+{i}?" for i in range(20)]    # First batch: cold cache    t0 = time.perf_counter()    llm_cached.generate(prompts[:5], params)    cold_time = (time.perf_counter() - t0) / 5    # Second batch: warm cache (prefix already computed)    t0 = time.perf_counter()    llm_cached.generate(prompts[5:10], params)    warm_time = (time.perf_counter() - t0) / 5    print(f"Cold cache TTFT (per request): {cold_time*1000:.1f} ms")    print(f"Warm cache TTFT (per request): {warm_time*1000:.1f} ms")    print(f"Speedup from prefix caching:   {cold_time/warm_time:.2f}x")except (ImportError, MemoryError, Exception) as e:    print(f"vLLM benchmark skipped: {e}")    print("Run on A10G/A100 for full vLLM prefix caching benchmark.")

## Key Takeaways1. **Prefix caching is memoization for attention**: identical token prefixes produce identical KV tensors.2. **TTFT scales with cached fraction**: 80% cached prefix = ~80% TTFT reduction.3. **Memory savings are Nx** where N = concurrent requests sharing the same prefix.4. **Exact match required**: one different token breaks the entire cache chain downstream.5. **Design prompts for cacheability**: static content first, dynamic content last.